# GOES X-ray CREP Analysis (47-year dataset)

This notebook analyses the 47-year synthetic GOES 1–8 Å time series (1975–2026)
and computes the CREP tensor components for the solar active-region system.

**Reference:** Velasco Herrera, V.M. et al. (2026), JGR Space Physics.

## Key outputs
- Power-law energy spectrum dN/dE ~ E^(-1.8)
- Permutation entropy of GOES flux (P component of CREP)
- AR(1) coefficient time series (critical slowing down)
- CREP Γ evolution over solar cycle

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from solar_flare_utac.goes_loader import GOESLoader
from solar_flare_utac.crep_solar import SolarCREP
from solar_flare_utac.constants import GAMMA_SOLAR

# Generate 47-year synthetic GOES time series
loader = GOESLoader(seed=42)
times, flux = loader.generate_synthetic(n_years=47, cadence_hours=1.0)

print(f'Generated {len(times):,} hourly GOES data points ({times[-1]/8760:.1f} years)')
print(f'Flux range: [{flux.min():.2e}, {flux.max():.2e}] W/m²')

In [ ]:
# Plot GOES flux time series
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

years = times / 8760
axes[0].semilogy(years, flux, lw=0.3, color='gold', alpha=0.7)
axes[0].set_ylabel('GOES 1–8 Å flux [W/m²]')
axes[0].set_xlabel('Year (from 1975)')
axes[0].set_title('Synthetic GOES X-ray Flux (47 years, seed=42)')

# Flare identification
flares = loader.identify_flares(flux, times, threshold_multiplier=50.0)
print(f'Identified {len(flares)} flare events')

# Power-law check of flare peak fluxes
if flares:
    peaks = np.array([f['peak_flux_W_m2'] for f in flares])
    peaks = peaks[peaks > 0]
    axes[1].loglog(*np.unique(np.sort(peaks), return_counts=True)[::-1],
                   '.', ms=3, color='orangered')
    axes[1].set_xlabel('Peak flux [W/m²]')
    axes[1].set_ylabel('N (>flux)')
    axes[1].set_title('Flare Flux CCDF (should follow power law α ≈ 1.8)')

plt.tight_layout()
plt.show()

In [ ]:
# Compute rolling permutation entropy (P component of CREP)
window = 720   # 30-day rolling window
step = 168     # 7-day step
pe_times, pe_values = [], []

for i in range(0, len(flux) - window, step):
    pe = loader.permutation_entropy(flux[i:i+window], m=4)
    pe_times.append(times[i + window//2] / 8760)
    pe_values.append(pe)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(pe_times, pe_values, lw=0.8, color='teal')
ax.axhline(0.97, ls='--', color='red', label='Nominal PE ≈ 0.97 → P ≈ 0.03')
ax.set_xlabel('Year (from 1975)')
ax.set_ylabel('Permutation entropy')
ax.set_title('Rolling permutation entropy of GOES flux (30-day window)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean permutation entropy: {np.mean(pe_values):.4f}')
print(f'Mean P component (1-PE):  {1-np.mean(pe_values):.4f}')

In [ ]:
# Rolling CREP Gamma over time
crep = SolarCREP(seed=42)

gammas, t_gamma = [], []
H_proxy = 0.15  # typical quiescent-active H

for i in range(0, len(flux) - window, step):
    flux_w = flux[i:i+window]
    cycle_phase = 2 * np.pi * (i / 8760) / 11.0
    state = crep.compute(H=H_proxy, dH_dt=0.003,
                         flux_series=flux_w,
                         solar_cycle_phase=cycle_phase)
    gammas.append(state.Gamma)
    t_gamma.append(times[i + window//2] / 8760)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_gamma, gammas, lw=0.8, color='orangered')
ax.axhline(GAMMA_SOLAR, ls='--', color='black', label=f'Γ_solar nominal={GAMMA_SOLAR:.4f}')
ax.set_xlabel('Year (from 1975)')
ax.set_ylabel('Γ (CREP)')
ax.set_title('Solar CREP Γ(t) over 47 years')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean Γ: {np.mean(gammas):.5f}  (target: {GAMMA_SOLAR:.5f})')